# 建立爬蟲核心工具箱

這一段示範的是一個基本的靜態網頁爬蟲流程，重點是把三個核心能力串起來：

- `requests`：向網站發送 HTTP 請求，取得原始 HTML
- `BeautifulSoup`：把 HTML 解析成可操作的樹狀結構
- `lxml`：作為 HTML 解析器，讓 `BeautifulSoup` 更穩定地抓取內容

此外也會用到 `time`、`json`、`re`，分別用來做延遲、資料儲存、字串模式匹配。這些模組會在之後的關卡中逐步派上用場。

In [23]:
# 若尚未安裝需要的套件，可在終端或 notebook 執行：
# !pip install requests beautifulsoup4 lxml

import requests         # 發送 HTTP 請求，用來從網站抓取網頁內容
from bs4 import BeautifulSoup as bs  # 解析 HTML / XML，將字串轉成可以操作的樹狀結構
import lxml             # 上課用的解析器（解析器需已安裝）
import time             # 取得目前時間，或讓程式暫停一段時間，做延遲，避免被 PTT 封鎖
import json             # 處理 JSON 格式的資料（例如 API 回傳或儲存設定）
import re               # 正規表達式

在開始抓取資料之前，我們先要模擬瀏覽器送出請求。這裡的目標是 PTT 股票版搜尋頁面，會帶入關鍵字 `0050` 來取得相關文章列表。

流程如下：

1. 準備搜尋 URL
2. 用 `requests.get()` 發送 GET 請求
3. 檢查 `response.status_code` 是否為 `200`
4. 用 `BeautifulSoup(..., 'lxml')` 解析 HTML
5. 透過 CSS 選擇器 `div.r-ent` 找出每一篇文章的區塊
6. 從每個文章區塊中取出日期與連結

這一步是整個爬蟲最基礎的入口，之後所有文章網址與內文都會依賴這個列表產生。

In [28]:
all_articles = []
# 構造 GET 請求網址：`?` 宣告參數開始，`&` 用來連接多個條件
for i in range(1,3):
    url = f'https://www.ptt.cc/bbs/Stock/search?page={i}&q=0050'

    # 發送 GET 請求到該網址
    # 提示：requests 會回傳一個 Response 物件
    res = requests.get(url)

    # 簡單檢查是否成功收到回應（HTTP 200 表示成功)
    print(f"第{i}頁，狀態碼為{res.status_code}")

    # 使用 BeautifulSoup 解析回傳的 HTML（使用 lxml 解析器，需要先安裝 lxml）
    # res.text 就是你在瀏覽器按右鍵「檢視網頁原始碼」看到的那些 HTML 內容  
    soup = bs(res.text, 'lxml')

    base_url = 'https://www.ptt.cc'
    # 使用 CSS 選擇器選取所有文章區塊 (定位、取值)
    # PTT 文章列表中每篇文章的外層容器，元素組成: <div class="r-ent"> ... </div>
    article_entries = soup.select('div.r-ent')
    for article in article_entries:
        # 開始篩選，日期<div class="date"> 9/04</div>
        date = article.select_one('div.date').text
        if date.strip().startswith(pattern):
            post_url = article.a['href']
            # 得到 'bbs/Stock/M.1785468093.A.C8B.html'
            # https://www.ptt.cc/bbs/Stock/M.1788514274.A.D51.html
            # 擷取下來的 url 不夠完整，補上伺服器的位置

            complete_url = base_url + post_url
            print(complete_url)
            all_articles.append(complete_url)

# 檢查點 : 印出數量看有沒有抓到東西
print(len(all_articles))

第1頁，狀態碼為200
https://www.ptt.cc/bbs/Stock/M.1790763476.A.DC8.html
https://www.ptt.cc/bbs/Stock/M.1790751166.A.CA5.html
https://www.ptt.cc/bbs/Stock/M.1790749765.A.5A2.html
https://www.ptt.cc/bbs/Stock/M.1790744249.A.8F8.html
https://www.ptt.cc/bbs/Stock/M.1790326816.A.3C3.html
https://www.ptt.cc/bbs/Stock/M.1790272552.A.0F6.html
https://www.ptt.cc/bbs/Stock/M.1790253392.A.F9A.html
https://www.ptt.cc/bbs/Stock/M.1790251417.A.5E8.html
https://www.ptt.cc/bbs/Stock/M.1790250495.A.875.html
https://www.ptt.cc/bbs/Stock/M.1790240011.A.5D5.html
https://www.ptt.cc/bbs/Stock/M.1790225128.A.68F.html
https://www.ptt.cc/bbs/Stock/M.1789975575.A.B65.html
https://www.ptt.cc/bbs/Stock/M.1789533838.A.FBA.html
https://www.ptt.cc/bbs/Stock/M.1789100508.A.A14.html
https://www.ptt.cc/bbs/Stock/M.1788968517.A.A15.html
第2頁，狀態碼為200
https://www.ptt.cc/bbs/Stock/M.1788514274.A.D51.html
https://www.ptt.cc/bbs/Stock/M.1788268644.A.AE9.html
17


接下來是爬蟲最關鍵的一步：定位資料位置。

在 PTT 的文章列表中，每篇文章都包在一個 `div.r-ent` 容器裡，這個容器內常見有日期、標題與連結資訊。我們會使用 BeautifulSoup 的 `select()` 方法，依照 CSS 選擇器直接抓出所有符合條件的文章區塊。

這裡的邏輯是：

- `soup.select('div.r-ent')`：取得整份列表中的所有文章卡片
- 每個文章卡片再往下取 `div.date`：拿到日期文字
- 再判斷這個日期是否符合我們要抓的月份條件
- 若符合，進一步取出 `a['href']`，獲得文章相對網址

也就是說，我們先把「文章列表」做成一個可操作的 Python 清單，接著再依序篩選出目標文章。

> 教學註記：這個函式是為了讓日期判斷自動化而設計的，它會依照系統目前時間，動態計算出當前月份。  
> 這個技巧在爬蟲中很常見，因為網站文章常常是依時間排序，而我們希望程式不必手動改月份。  
> 
> 核心機制：  
> 1. `datetime.now()` 建立當前時間物件  
> 2. 使用 `.month` 取出月份的整數值  
> 3. 回傳結果，供後續篩選上個月文章使用  
> 
> 例如：現在是 1 月，程式會判斷上個月是 12 月；如果現在是 8 月，則上個月就是 7 月。

In [25]:
from datetime import datetime

'''
練習如何從datetime物件動態提取月份，並傳遞給month使用
'''
def month_extract():
    # 1. 獲取當前時間物件 (Instantiation)
    # 這是動態的源頭，每次執行時都會抓取系統當下時間
    current_time = datetime.now()

    # 2. 提取屬性 (Attribute Extraction)
    # 找month屬性 => .month
    # 注意 : 屬性存取，不需要括號 ()
    This_month = current_time.month

    # 3.  驗證輸出
    # print(f"系統偵測時間:{current_time}")
    # print(f"提取月份:{This_month}")
    return This_month
if __name__ == "__main__":
    print(month_extract())
    

10


### 任務：實作「上月文章過濾器」

這一段的目的是從全部文章中只保留「上個月」的文章，這樣可以避免抓到太多不相關的近期文章。

實作邏輯如下：

1. 先用 `month_extract()` 取得目前月份
2. 若目前是 1 月，則上個月應該是 12 月
3. 否則，上個月就是當前月份減 1
4. 建立 `pattern = f"{last_month}/"`，用來比對文章日期字串
5. 使用 `date_text.strip().startswith(pattern)` 檢查日期是否符合格式

這裡非常重要的一點是 `startswith('/')` 的寫法。它會要求日期字串以 `"{month}/"` 開頭，例如 `"1/"`、`"2/"`、`...`、`"12/"`，這樣能避免把 `1` 誤判成 `11` 或 `12` 的前綴問題。

也就是說，我們不是粗略比對數字，而是精準比對日期格式，才能正確過濾出上個月發文。

In [26]:
# month_extract() 已經取得當前月份
# 為甚麼需要寫這個if判斷? 請學員討論
if int(month_extract()) == 1:
    last_month = 12
else:
    last_month = month_extract() -1

# Pattern(匹配的格式)，我們預期要匹配到"1/"、"2/"、...、"12/"
# f-string（格式化字串）：在字串前加 f，使用 {變數} 直接嵌入變數值，簡潔替代 str.format()
pattern = f"{last_month}/"

#.strip():去除前後空白


### 任務：打造網址收集器（URL Collector）

已經篩選出上個月發表的文章後，下一步是把它們的文章連結收集起來，方便之後逐篇開啟並抓取內文。

這個步驟的重點有三個：

1. 建立一個空列表 `target_urls`，用來儲存最終網址
2. 定義 `base_url = "https://www.ptt.cc"`，作為網站根網址
3. 對每一篇符合條件的文章，取出 `a['href']`，再將其補成完整 URL

為什麼需要 `base_url`？因為 PTT 的 `href` 通常是相對路徑，例如：

- `bbs/Stock/M.1234567890.A.123.html`

這種路徑不能直接拿來請求，必須補上域名，變成：

- `https://www.ptt.cc/bbs/Stock/M.1234567890.A.123.html`

最後，我們把整理好的完整網址一個一個 `append()` 到 `target_urls`，這樣整個文章清單就能再往下一步進行。

### 任務：儲存多篇文章資料

當我們已經有一批目標文章網址時，下一步就是進入每篇文章頁面，把關鍵資訊蒐集起來。

這裡使用的是一種很常見的資料整理方式：

- `articles_data = []`：總表，裝所有文章資料
- 每篇文章都用一個字典 `article_info` 表示
- 字典中至少記錄標題與網址，例如：

```python
article_info = {
    'title': title,
    'url': target_url
}
```

在迴圈中，對每個網址送出一個請求，解析 HTML，擷取該文章的標題、時間、內容、留言等資訊，最後使用 `articles_data.append(article_info)` 將它存進總列表裡。

這樣做的好處是：後續如果要做資料分析、匯出 CSV、做文字探勘，都能直接對 `articles_data` 這份清單進行處理，而不是每篇文章各自散落。

In [27]:
from time import sleep

# 建立大列表
articles_data = []

# 建立「單篇文章」的字典(Dict)
# all_articles
# 把「單篇文章」的字典裝進大列表裡面
for url in all_articles:
    try:
        # span.article-meta-value -> 標題是第三個 ; 時間是第四個
        res = requests.get(url)

        #HTML 原始碼 -> res.text
        soup = bs(res.text, 'lxml')
        # 取標題
        meta_value = soup.select('span.article-meta-value')
        if len(meta_value) >=3:
            title = meta_value[2].text

        # 取時間
        if len(meta_value)>=4:
            time = meta_value[3].text

        # 取留言
        css_selector = 'span.f3.push-content'
        comments = soup.select(css_selector)
        comment_list = []

        for comment in comments:
            review = comment.text
            comment_list.append(review)

        # 取內容?


        # PTT (批踢踢實業坊)，它的HTML結構並非現代化的語意標籤 (Semantic HTML)，而是由舊式BBS系統轉換而來
        # 這導致「內文」並不是包在一個乾淨的 <p> 或 <article-body> 標籤中
        # 而是作為「文字節點 (Text Node)」散落在 #main-content元素內，與標題、時間、推文混雜在一起
        # 這裡我們不需要保留「標題列」或「時間」，我們使用beautifulsoup的 decomposed() 方法將不需要的元素去除
        
        # 步驟一 : 鎖定主內容元素
        main_content = soup.select_one('#main-content')

        # 確保程式不會在找不到網頁內容時不會崩潰 (Crash)
        # 使用 if xxx: 若遇到xxx=None 不會停止執行而是「優雅地跳過」
        if main_content:
            # 步驟二: 移除不要的元素
            # 批次處理：利用 CSS Selector 的逗號 , 功能，一次選取所有不要的類別（Meta + Push），一個迴圈解決，不需要寫好幾個迴圈。
            # 不要的元素(1) : .article-metaline (標題、作者、時間)
            # 不要的元素(2) : .article-metaline-right (看板名稱)
            # 不要的元素(3) : span (零散資訊)
            # 不要的元素(4) : div.push (推文區塊)
            unwanted_elements = main_content.select('.article-metaline, .article-metaline-right, span, div.push')
            #步驟三: 執行銷毀
            for ele in unwanted_elements:
                ele.decompose()

            # 複習 .strip(), replace() 清理 main_content 內容
            clean_content = main_content.text.strip().replace('\n','')

            article_info= {
                'title':title,
                'time':time,
                'comment': comment_list,
                'content': clean_content,
                'url':url
            }
            articles_data.append(article_info)
            print(f"目前爬取{title}文章")
            sleep(1)

    except Exception as e:
        print(f"{url}爬取失敗，錯誤原因:{e}")
from pprint import pprint
pprint(articles_data)


目前爬取Re: [新聞] 0050每月定期定額3000元「分3次扣」有必文章
目前爬取Re: [新聞] 0050每月定期定額3000元「分3次扣」有必文章
目前爬取Re: [新聞] 0050每月定期定額3000元「分3次扣」有必文章
目前爬取[新聞] 0050每月定期定額3000元「分3次扣」有必 文章
目前爬取Re: [請益] 全押0050的人為何不全壓00631L ??文章
目前爬取Re: [請益] 全押0050的人為何不全壓00631L ??文章
目前爬取Re: [請益] 全押0050的人為何不全壓00631L ??文章
目前爬取Re: [請益] 全押0050的人為何不全壓00631L ??文章
目前爬取Re: [請益] 全押0050的人為何不全壓00631L ??文章
目前爬取[請益] 全押0050的人為何不全壓00631L ??文章
目前爬取[請益] 台股只投資0050有需要再搭配其他嗎？文章
目前爬取[標的] 股市大跌應該要加碼0050/0052?文章
目前爬取[新聞] 少年股神心頭好曝光！　大學生最愛0050、文章
目前爬取[新聞] 0050該等大跌還是持續買？「買綠不買紅」文章
目前爬取[新聞] 0050已賺1800萬！郭哲榮預言台股「3年奔1文章
目前爬取[新聞] 0050換股名單出爐！藥華藥卡位成功、世芯文章
目前爬取[新聞] 郭哲榮砸1億買0050大賺1600萬！9月要賣嗎文章
[{'comment': [': 收入少更要投資',
              ': 收入少就提升本業 台積工程師賺一年收入 3000投資10',
              ': 年也贏不了',
              ': 人比人氣死人，要是按2樓邏輯那直接躺平就好。定期',
              ': 定額要，努力提高收入也要。',
              ': 分散3筆 感覺是分散風險 實際上也是提高風險',
              ': 分三筆都買到高點www',
              ': 每日買入？'],
  'content': '其實什麼報酬率差不多啦3000塊分三次最大的問題是單筆金額太低投入會少很多假設好算一點的股價101來說1000塊就是買9股909一個月27股27273000塊是買29股2929股價越高或單次金額越低這

In [ ]:
json.dump(articles_data)